In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

In [ ]:
#!uv pip install datasets
df = pd.read_parquet("hf://datasets/juliensimon/sentry-impact-risk/data/sentry_impact_risk.parquet")
df.head()

In [ ]:
df.info()

In [ ]:
df.isna().sum()
#df.dropna()

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # Force CPU only
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df_reg = df.copy()
df_reg = df_reg.dropna() 
df_reg['log_impact_probability'] = np.log10(df_reg['impact_probability'])

# Data loading and preprocessing
features = df.select_dtypes(include=['number']).columns.tolist()
features.remove('impact_probability')
X = df_reg[features]
y = df_reg['log_impact_probability']
scaler = StandardScaler()

X_processed = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_processed, y, test_size=0.2, random_state=42)

In [ ]:
# Model definition
model = tf.keras.Sequential([
    layers.Input(shape=[len(X.keys())]),
    layers.Dense(64, activation='relu'),
    layers.Dense(64, activation='relu'),
    layers.Dense(1)
])

# Loss, optimizer, and metrics
optimizer = tf.keras.optimizers.Adam()#.RMSprop(0.001)
model.compile(optimizer=optimizer, loss='mse', metrics=['mae']) #metrics=['mae', 'mse']

# Training
model.fit(X_train, y_train, epochs=10, batch_size=64, verbose=2)

# Evaluation
test_loss, test_mae = model.evaluate(X_test, y_test,batch_size=30, verbose=0)
print(f"Test MSE (Loss): {test_loss:.4f}")
print(f"Test MAE (Mean Absolute Error): {test_mae:.4f}")
